In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

#import personnal tools
import sys
sys.path.append('../tools/')
from info import *
from imports import *
from tools_generic import *
from events import *

# Load files

In [ ]:
site_list=["d17","d47","d85","dmc"]
file_start_date = '20241201'
file_end_date = '20260630'

golden_start_date="2025-01-01"
golden_end_date="2025-03-31"
# golden_end_date="2025-05-31"

In [ ]:
data = {}
daily_data = {}

In [ ]:
data = load_flowcapt_data(data, site_list, file_start_date, file_end_date)
data = load_spc_data(data, site_list, file_start_date, file_end_date)
data = load_surf_data(data, site_list, file_start_date, file_end_date)
data = load_wind_data(data, site_list, file_start_date, file_end_date) # WIND loading includes the early WIND_BEGINNING files by default.
data = load_mrr_precip_fraction(
    data,
   
 site=["d17", "d47"],
    file_start_date=golden_start_date.replace("-", ""),
    file_end_date=golden_end_date.replace("-", ""),
    timestep="30min",
    dbz_threshold=1.0,
    altitude_range=(300, 1000),
    variable_name="precip_fraction_mrr",
)

In [ ]:
# Extract period of interest (golden month)
data = filter_datasets_golden(
    data, start_date=golden_start_date, end_date=golden_end_date
 )

In [ ]:
# create daily
daily_data = create_daily_data(data)

## Export to csv for AWACA db

In [ ]:
data['d17']

### FlowCapt

In [ ]:
# Load available FlowCapt files separately from the golden-month analysis data.
export_start = "2025-02-13 00:00:00"
export_end = "2025-02-19 23:00:00"
available_file_start = "20241201"
available_file_end = "20260630"  # Latest FlowCapt file currently available.
export_sites = ["d17", "d47", "d85", "dmc"]
export_site_prefix = {
    "d17": "D17",
    "d47": "D47",
    "d85": "D85",
    "dmc": "DMC",
}
flowcapt_units = {
    "FluxMin1": "g_per_m2_per_s",
    "FluxMean1": "g_per_m2_per_s",
    "FluxMax1": "g_per_m2_per_s",
    "FluxStd1": "g_per_m2_per_s",
    "FluxSum1": "g_per_m2",
    "FluxMin2": "g_per_m2_per_s",
    "FluxMean2": "g_per_m2_per_s",
    "FluxMax2": "g_per_m2_per_s",
    "FluxStd2": "g_per_m2_per_s",
    "FluxSum2": "g_per_m2",
    "WindMin1": "m_per_s",
    "WindMean1": "m_per_s",
    "WindMax1": "m_per_s",
    "WindMin2": "m_per_s",
    "WindMean2": "m_per_s",
    "WindMax2": "m_per_s",
    "Hagl": "m",
}

flowcapt_export_source = load_flowcapt_data(
    {}, export_sites, available_file_start, available_file_end
)
hourly_index = pd.date_range(
    start=export_start,
    end=export_end,
    freq="1h",
    name="datetime",
)

excluded_variables = {"RECORD_flowcapt", "DT_flowcapt"}
variable_names = sorted(
    {
        variable
        for ds in flowcapt_export_source.values()
        for variable in ds.data_vars
        if variable not in excluded_variables
    }
)
if not variable_names:
    raise ValueError("No FlowCapt data variables were loaded")

flowcapt_export_ds = xr.Dataset(coords={"datetime": hourly_index})
for site in export_sites:
    prefix = export_site_prefix[site]
    ds = flowcapt_export_source.get(site)
    if ds is not None and "time" in ds.coords:
        site_hourly = (
            ds.sel(time=slice(export_start, export_end))
            .resample(time="1h", closed="left", label="left")
            .mean()
            .reindex(time=hourly_index)
        )
    else:
        site_hourly = None

    for variable in variable_names:
        base_name = variable.removesuffix("_flowcapt")
        unit = flowcapt_units.get(base_name)
        if unit is None:
            raise ValueError(f"No unit is defined for FlowCapt variable {base_name!r}")
        column = f"{prefix}_{base_name}_{unit}"
        if site_hourly is not None and variable in site_hourly.data_vars:
            flowcapt_export_ds[column] = (
                "datetime",
                site_hourly[variable].values,
            )
        else:
            flowcapt_export_ds[column] = (
                "datetime",
                pd.Series(index=hourly_index, dtype=float).to_numpy(),
            )
        flowcapt_export_ds[column].attrs["units"] = unit

flowcapt_export_df = flowcapt_export_ds.to_dataframe().reset_index()
flowcapt_export_df["datetime"] = flowcapt_export_df["datetime"].dt.strftime(
    "%Y-%m-%d %H:%M:%S"
)
assert len(flowcapt_export_df) == len(hourly_index)
assert flowcapt_export_df["datetime"].iloc[0] == export_start
assert flowcapt_export_df["datetime"].iloc[-1] == export_end
assert len(flowcapt_export_df) == 168
assert not any(column.startswith("DDU_") for column in flowcapt_export_df.columns)
assert not any("RECORD_flowcapt" in column or "DT_flowcapt" in column for column in flowcapt_export_df.columns)
assert not any("_flowcapt" in column for column in flowcapt_export_df.columns)
assert all("[" not in column and "]" not in column for column in flowcapt_export_df.columns)

coverage_by_site = {
    prefix: int(
        flowcapt_export_df[
            [column for column in flowcapt_export_df if column.startswith(f"{prefix}_")]
        ].notna().any(axis=1).sum()
    )
    for prefix in export_site_prefix.values()
}
print(f"Prepared {len(flowcapt_export_df):,} hourly rows, {len(flowcapt_export_df.columns) - 1} data columns")
print("Non-empty hourly rows by site:", coverage_by_site)
flowcapt_export_df.head()

In [ ]:
# Export the prepared hourly table to CSV in the current working directory.
flowcapt_export_csv = "exports/flowcapt_awaca_hourly_20250213_20250219.csv"
flowcapt_export_df.to_csv(flowcapt_export_csv, index=False)
print(f"Exported {len(flowcapt_export_df):,} hourly rows to {flowcapt_export_csv}")

### SPC

In [ ]:
# Load SPC files separately from the golden-month analysis data.
spc_export_start = "2025-02-13 00:00:00"
spc_export_end = "2025-02-19 23:00:00"
spc_available_file_start = "20241201"
spc_available_file_end = "20260630"
spc_export_sites = ["d17", "d47", "d85"]
spc_site_prefix = {"d17": "D17", "d47": "D47", "d85": "D85"}
spc_units = {
    "temperature": "celsius",
    "snowflux": "g_per_m2_per_s",
    "particles_per_sec": "per_s",
    "voltage": "mV",
}
spc_variable_longnames = {
    "temperature": "Air Temperature",
    "snowflux": "Snow Mass Flux",
    "particles_per_sec": "Snow Particle Count per Second",
    "voltage": "Sensor Battery Voltage",
}

spc_export_source = load_spc_data(
    {}, spc_export_sites, spc_available_file_start, spc_available_file_end
)
spc_hourly_index = pd.date_range(
    start=spc_export_start,
    end=spc_export_end,
    freq="1h",
    name="datetime",
)
spc_variable_names = ["temperature_spc", "snowflux_spc", "particles_per_sec_spc", "voltage_spc"]

spc_export_ds = xr.Dataset(coords={"datetime": spc_hourly_index})
for site in spc_export_sites:
    prefix = spc_site_prefix[site]
    ds = spc_export_source.get(site)
    if ds is not None and "time" in ds.coords:
        site_hourly = (
            ds.sel(time=slice(spc_export_start, spc_export_end))
            .resample(time="1h", closed="left", label="left")
            .mean()
            .reindex(time=spc_hourly_index)
        )
    else:
        site_hourly = None

    for variable in spc_variable_names:
        base_name = variable.removesuffix("_spc")
        unit = spc_units[base_name]
        column = f"{prefix}_{base_name}_{unit}"
        if site_hourly is not None and variable in site_hourly.data_vars:
            values = site_hourly[variable].values
        else:
            values = pd.Series(index=spc_hourly_index, dtype=float).to_numpy()
        spc_export_ds[column] = ("datetime", values)
        spc_export_ds[column].attrs.update(
            {"units": unit, "long_name": spc_variable_longnames[base_name]}
        )

spc_export_df = spc_export_ds.to_dataframe().reset_index()
spc_export_df["datetime"] = spc_export_df["datetime"].dt.strftime(
    "%Y-%m-%d %H:%M:%S"
)
assert len(spc_export_df) == 168
assert spc_export_df["datetime"].iloc[0] == spc_export_start
assert spc_export_df["datetime"].iloc[-1] == spc_export_end
assert len(spc_export_df.columns) == 13
expected_spc_columns = {
    f"{prefix}_{variable}_{unit}"
    for prefix in spc_site_prefix.values()
    for variable, unit in spc_units.items()
}
assert set(spc_export_df.columns[1:]) == expected_spc_columns

spc_coverage_by_site = {
    prefix: int(
        spc_export_df[
            [column for column in spc_export_df if column.startswith(f"{prefix}_")]
        ].notna().any(axis=1).sum()
    )
    for prefix in spc_site_prefix.values()
}
print(f"Prepared {len(spc_export_df):,} hourly rows, {len(spc_export_df.columns) - 1} data columns")
print("Non-empty hourly rows by site:", spc_coverage_by_site)
spc_export_df.head()

In [ ]:
# Export the prepared hourly SPC table to CSV in the current working directory.
spc_export_csv = "spc_awaca_hourly_20250213_20250219.csv"
spc_export_df.to_csv(spc_export_csv, index=False)
print(f"Exported {len(spc_export_df):,} hourly rows to {spc_export_csv}")

# Stats

In [ ]:
variable = 'FluxMean1'
compute_variable_stats(data, variable)

In [ ]:
var = "FluxMean1_flowcapt"
# plot_binned_distribution(data, var, bin_number=30, min_value=0, max_value=200)


In [ ]:
var = "snowflux_spc"
# plot_binned_distribution(data, var, bin_number=30, min_value=0, max_value=200)


In [ ]:
variables = ["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc", "wspd1_wind", "wdir_wind", "precip_fraction_mrr"]
plot_monthly_availability_table(
    sensor_datasets=data,
    variables=variables,
    sites=['d17', 'd47', 'd85'],
    figsize=(16, len(variables)),
)

In [ ]:
variables = ["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc", "wspd1_wind", "wdir_wind", "precip_fraction_mrr"]
plot_daily_availability_table(
    sensor_datasets=data,
    variables=variables,
    sites=['d17', 'd47', 'd85'],
    figsize=(16, len(variables)),
)

# Basic plotting

## Time plots

In [ ]:
variables = ["FluxMean1_flowcapt", "FluxMean2_flowcapt"]#, "snowflux_spc"]
variables = ['snowflux_spc']

plot_per_site_multiple_vars(
    data,
    variables,
    # sites=["d17", "d47", "d85"],
    sites = ["d85"],
    figsize=(15, 5),
    ymax=5,
)


In [ ]:
variables = ["wdir_wind", "wspd1_wind"]#, 'precip_fraction_mrr']

# plot_per_var_multiple_sites(
#     sensor_datasets=data,
#     variables=variables,
#     sites=["d17", "d47", "d85"],
#     ymin=[0, 0, -0.05],
#     ymax=[360, 26, 1.05],
# )


## Scatters

In [ ]:
var1 = "wdir_wind"
var2 = "wspd1_wind"
site1 = "d17"
site2 = "d17"
plot_bivariate_scatter(
    data,
    var1=var1,
    var2=var2,
    site1=site1,
    site2=site2,
    show_corr=False,
    show_fit=False,
    min_val=[0, 0],
    max_val=[360, 26],
    figsize=(7, 6),
)


In [ ]:
var1 = "FluxMean2_flowcapt"
var2 = "snowflux_spc"
var3 = "wspd1_wind"
site1 = "d47"
site2 = "d47"
site3 = "d47"
plot_trivariate_scatter(
    data,
    var1=var1,
    var2=var2,
    var3=var3,
    site1=site1,
    site2=site2,
    site3=site3,
    min3=10,
    max3=20,
    max_val=[300, 300],
    figsize=(7, 6),
    show_oneone=True,
)


# Events

In [ ]:
detector = EventDetector(
    threshold=1.0,
    min_timesteps=8,
    buffer_timesteps=0,
)

sampled_data = create_resampled_data(data, "30min")
additional_variables = [
    "FluxMean1_flowcapt",
    "Hagl_flowcapt",
    "snowflux_spc",
    "wspd1_wind",
    "wdir_wind",
    "T1_surf",
    "RH1_surf",
    'precip_fraction_mrr'
]
collection = detector.detect_events(
    sampled_data,
    variable="FluxMean2_flowcapt",
    additional_variables=additional_variables,
)
collection_d17 = collection.get_site("d17")
collection_d47 = collection.get_site("d47")
collection_d85 = collection.get_site("d85")

non_collection = detector.detect_non_events(
    sampled_data,
    variable="FluxMean2_flowcapt",
    additional_variables=additional_variables,
)
non_collection_d17 = non_collection.get_site("d17")
non_collection_d47 = non_collection.get_site("d47")
non_collection_d85 = non_collection.get_site("d85")


In [ ]:
collection_d17.to_catalog(variables=["FluxMean2"]).head()

In [ ]:
non_collection_d17.to_catalog(variables=["FluxMean2"]).head()

In [ ]:
print_duration_stats(collection)

In [ ]:
print_duration_stats(collection_d17)
print_duration_stats(collection_d47)
print_duration_stats(collection_d85)

In [ ]:
print_integrated_flux_stats(collection, variable='FluxMean2_flowcapt')
print_integrated_flux_stats(collection_d17, variable='FluxMean2_flowcapt')
print_integrated_flux_stats(collection_d47, variable='FluxMean2_flowcapt')

In [ ]:
plot_diurnal_start_distribution(collection_d17)

In [ ]:
varlist = ["FluxMean2_flowcapt", "wspd1_wind", 
        #    "wdir_wind", 
        'precip_fraction_mrr']
plot_events_vs_nonevents_chronological(
    collection_d17,
    non_collection_d17,
    variables=varlist,
    figsize=(15, len(varlist) * 3),
)
plot_events_vs_nonevents_chronological(
    collection_d47,
    non_collection_d47,
    variables=varlist,
    figsize=(15, len(varlist) * 3  ),
)

plot_events_vs_nonevents_chronological(
    collection_d85,
    non_collection_d85,
    variables=varlist,
    figsize=(15, len(varlist) * 3),
)


### Time series

In [ ]:
plot_event_collection_traces(
    collection=collection_d17,
    variable="wspd1_wind",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.8,
    show_mean=False,
)


In [ ]:
plot_event_collection_traces(
    collection=non_collection_d17,
    variable="FluxMean2_flowcapt",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.7,
)


In [ ]:
plot_event_collection_traces(
    collection=collection_d47,
    variable="wspd1_wind",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.7,
    show_mean=False,
)
